# Agent-R PPO Training
**Step 5 (Part 2):** 200 PPO steps on Qwen2.5-3B-Instruct + SFT adapter

**Input:** `/kaggle/input/agent-r-sft-adapter/` (your saved SFT adapter dataset)

**Output:** `agent-r-v0` — final Agent-R checkpoint, ready for co-evolution

In [ ]:
!pip install -q transformers==4.47.0 trl==0.12.0 peft==0.14.0 datasets accelerate bitsandbytes sentencepiece

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset

BASE_MODEL   = "/kaggle/input/datasets/ebadshabbir/qwen2-5-3b-instruct"
SFT_ADAPTER  = "/kaggle/input/agent-r-sft-adapter"   # your saved dataset
OUTPUT_DIR   = "/kaggle/working/agent_r_v0"
CHECKPOINT_DIR = "/kaggle/working/agent_r_ppo_checkpoints"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Setup complete")

## 1. Load Tokenizer + Model (base + SFT adapter)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # required for PPO generation

# Load base model, then attach SFT LoRA adapter, then wrap with value head for PPO
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

# Load the SFT adapter on top
base_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER, is_trainable=True)
base_model = base_model.merge_and_unload()   # merge SFT weights before adding value head

# Add a NEW LoRA on top for PPO updates + value head
ppo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = AutoModelForCausalLMWithValueHead.from_pretrained(
    base_model,
    peft_config=ppo_lora_config
)

print(f"Model loaded. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 2. Reward Function
Checks if the model's answer matches the GSM8K ground truth number.
Returns +1.0 for correct, -0.5 for wrong, -1.0 for no number found.

In [ ]:
def extract_number(text: str):
    """Pull the last number from a string (GSM8K answers end with #### <number>)"""
    # First try to find #### pattern (GSM8K format)
    match = re.search(r'####\s*([\-\d\.\,]+)', text)
    if match:
        return match.group(1).replace(',', '').strip()
    # Fallback: last number in text
    numbers = re.findall(r'[\-\d]+\.?\d*', text)
    return numbers[-1] if numbers else None


def compute_reward(response: str, ground_truth: str) -> float:
    """Returns reward tensor for a single response."""
    pred = extract_number(response)
    true = extract_number(ground_truth)

    if pred is None:
        return -1.0   # no answer produced
    if pred == true:
        return 1.0    # correct
    # Partial credit: if the number is close (within 1%) give small reward
    try:
        if abs(float(pred) - float(true)) / (abs(float(true)) + 1e-8) < 0.01:
            return 0.5
    except:
        pass
    return -0.5   # wrong answer


# Quick sanity check
print(compute_reward("The answer is #### 42", "#### 42"))   # expect 1.0
print(compute_reward("I think it is 100",     "#### 42"))   # expect -0.5
print(compute_reward("I have no idea",        "#### 42"))   # expect -1.0

## 3. Dataset — GSM8K (same as SFT, keeps distribution consistent)

In [ ]:
dataset = load_dataset("gsm8k", "main", split="train")

def format_prompt(example):
    """Format as a reasoning prompt — no answer included (PPO generates this)"""
    return {
        "query": f"Solve step by step:\n{example['question']}\nSolution:",
        "answer": example["answer"]
    }

dataset = dataset.map(format_prompt)
dataset = dataset.select(range(min(2000, len(dataset))))  # cap for speed

print(f"Dataset size: {len(dataset)}")
print("Sample prompt:", dataset[0]["query"][:200])

## 4. PPO Config

In [ ]:
ppo_config = PPOConfig(
    model_name=BASE_MODEL,
    learning_rate=1e-5,
    batch_size=8,           # 8 questions per PPO step
    mini_batch_size=2,      # process 2 at a time on T4
    gradient_accumulation_steps=4,
    ppo_epochs=4,           # inner PPO epochs per batch
    max_grad_norm=0.5,
    kl_penalty="kl",
    init_kl_coef=0.2,       # KL penalty against SFT reference
    adap_kl_ctrl=True,      # adaptive KL control
    target_kl=6.0,
    gamma=1.0,
    lam=0.95,
    cliprange=0.2,
    cliprange_value=0.2,
    vf_coef=0.1,
    seed=42,
    log_with=None,
)

print("PPO config set. Steps to run: 200")
print(f"Questions per step: {ppo_config.batch_size} → Total questions: {200 * ppo_config.batch_size}")

## 5. PPO Trainer Init

In [ ]:
def tokenize_query(sample):
    sample["input_ids"] = tokenizer.encode(sample["query"])
    return sample

dataset = dataset.map(tokenize_query)
dataset.set_format(type="torch", columns=["input_ids"])

def collator(data):
    return {key: [d[key] for d in data] for key in data[0]}

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    ref_model=None,   # trl uses the frozen SFT weights inside the PEFT model as reference
    tokenizer=tokenizer,
    dataset=dataset,
    data_collator=collator,
)

print("PPO Trainer ready")

## 6. Generation Config

In [ ]:
generation_kwargs = {
    "min_length": -1,
    "top_k": 0,
    "top_p": 1.0,
    "do_sample": True,
    "temperature": 0.7,
    "pad_token_id": tokenizer.eos_token_id,
    "max_new_tokens": 256,   # enough for GSM8K step-by-step solutions
}

## 7. PPO Training Loop — 200 Steps
Checkpoints every 50 steps. Early stopping if mean reward plateaus for 20 steps.

In [ ]:
from torch.utils.data import DataLoader
import json

MAX_STEPS = 200
CHECKPOINT_EVERY = 50
EARLY_STOP_PATIENCE = 20   # stop if no improvement for 20 steps

reward_log = []
best_mean_reward = -999
no_improve_count = 0

# Store answers alongside dataset for reward computation
answer_list = dataset["answer"]

print(f"Starting PPO training for {MAX_STEPS} steps...")
print("-" * 60)

for step, batch in enumerate(ppo_trainer.dataloader):
    if step >= MAX_STEPS:
        break

    query_tensors = batch["input_ids"]

    # ── Generate responses ──
    response_tensors = ppo_trainer.generate(
        query_tensors,
        return_prompt=False,
        **generation_kwargs
    )

    # Decode responses
    batch_responses = [tokenizer.decode(r.squeeze(), skip_special_tokens=True)
                       for r in response_tensors]

    # ── Compute rewards ──
    # Get ground truth answers for this batch
    # (dataloader gives indices implicitly via step * batch_size)
    start_idx = (step * ppo_config.batch_size) % len(answer_list)
    batch_answers = [
        answer_list[(start_idx + i) % len(answer_list)]
        for i in range(len(batch_responses))
    ]

    rewards = [
        torch.tensor(compute_reward(resp, ans), dtype=torch.float32)
        for resp, ans in zip(batch_responses, batch_answers)
    ]

    mean_reward = sum(r.item() for r in rewards) / len(rewards)
    reward_log.append({"step": step + 1, "mean_reward": mean_reward})

    # ── PPO update ──
    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

    # ── Logging ──
    if (step + 1) % 10 == 0:
        print(f"Step {step+1:3d}/200 | Mean reward: {mean_reward:.3f} | "
              f"KL: {stats.get('objective/kl', 0):.3f} | "
              f"Policy loss: {stats.get('ppo/loss/policy', 0):.4f}")

    # ── Checkpoint every 50 steps ──
    if (step + 1) % CHECKPOINT_EVERY == 0:
        ckpt_path = f"{CHECKPOINT_DIR}/step_{step+1}"
        ppo_trainer.model.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)
        print(f"  ✓ Checkpoint saved: {ckpt_path}")

    # ── Early stopping ──
    if mean_reward > best_mean_reward + 0.01:
        best_mean_reward = mean_reward
        no_improve_count = 0
    else:
        no_improve_count += 1

    if no_improve_count >= EARLY_STOP_PATIENCE and step > 50:
        print(f"\nEarly stopping at step {step+1} — no improvement for {EARLY_STOP_PATIENCE} steps.")
        break

print("-" * 60)
print(f"Training complete. Best mean reward: {best_mean_reward:.3f}")

# Save reward log
with open("/kaggle/working/ppo_reward_log.json", "w") as f:
    json.dump(reward_log, f, indent=2)
print("Reward log saved.")

## 8. Save Final Model — agent-r-v0

In [ ]:
ppo_trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Agent-R-v0 saved to {OUTPUT_DIR}")

# List output files
for f in os.listdir(OUTPUT_DIR):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"  {f}  ({size_mb:.2f} MB)")

## 9. Quick Sanity Check — is Agent-R reasoning?
Run 3 GSM8K questions and print responses.

In [ ]:
test_questions = [
    "Janet has 3 brothers and 2 sisters. How many siblings does she have?",
    "A store sells apples for $2 each and oranges for $3 each. If you buy 4 apples and 2 oranges, how much do you spend?",
    "A train travels 60 miles per hour. How far does it travel in 2.5 hours?",
]

model.eval()
with torch.no_grad():
    for q in test_questions:
        prompt = f"Solve step by step:\n{q}\nSolution:"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.pretrained_model.device)
        outputs = model.pretrained_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,   # greedy for eval
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        print(f"Q: {q}")
        print(f"A: {response.strip()}")
        print("-" * 50)

## ✅ Done!
**Next steps:**
1. Go to Kaggle output → Save `agent_r_v0` folder as dataset named `agent-r-v0` (private)
2. Step 5 complete ✓
3. Next: **Step 6 — Train Agent-D** (Qwen → PubMed pretraining → MedQA/PubMedQA SFT)